
notebook_02_entity_extraction.ipynb

Phase 2 — Entity Extraction

Input:  ProductCatalog_TRANSLATED_v2.xlsx

Output: product_entities.json


In [2]:
# ── CELL 1 — Imports ─────────────────────────────────────────────────────────
import os
import re
import json
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.environ.get('OPENAI_API_KEY', '')

if not api_key:
    import getpass
    api_key = getpass.getpass('OpenAI API key: ')

client = OpenAI(api_key=api_key)
MODEL  = 'gpt-5.4-mini'
print(f'Model: {MODEL}')

Model: gpt-5.4-mini


In [10]:
# ── CELL 2 — Load translated file
from google.colab import files
uploaded = files.upload()   # upload ProductCatalog_TRANSLATED_v2.xlsx

df = pd.read_excel('ProductCatalog_TRANSLATED_v2.xlsx')
print(f'Loaded: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(2)

Saving ProductCatalog_TRANSLATED_v2.xlsx to ProductCatalog_TRANSLATED_v2.xlsx
Loaded: (114, 13)
Columns: ['Product ID', 'Product Name', 'English Name', 'Product Type', 'Usage Instructions', 'Dilution Method', 'Target Crops', 'Specification', 'Data Source', 'Main Ingredients', 'Usage Method and Dosage', 'ingredient_source', 'product_name_cn']


,Product ID,Product Name,English Name,Product Type,Usage Instructions,Dilution Method,Target Crops,Specification,Data Source,Main Ingredients,Usage Method and Dosage,ingredient_source,product_name_cn
0,AF0001,Citrus Junliqing,Citrus Bacteria Clear,Growth Promoter,Core control targets (diseases throughout the ...,1 g per 0.5 L water,Registered crops: Chili pepper. Practice has s...,50g/500g,Original Excel Table + Image Labels,"Active strains: Bacillus subtilis, Bacillus li...","During the crop growth period, uniformly dilut...",NaN,柑橘菌立清
1,AF0002,"Allium Bacteria Clear for Onion, Ginger, and G...",Onion-Ginger-Garlic Bacteria Clear,Growth Promoter,Core control targets (diseases throughout the ...,1 g per 0.5 L water,Registered crops: Chili pepper. Practice has s...,50g/500g,Excel original table + image labels,"Active strains: Bacillus subtilis, Bacillus la...","During the crop growth period, uniformly dilut...",NaN,葱姜蒜菌立清


In [11]:
# ── CELL 3 — Helper: safe text ───────────────────────────────────────────────
def safe_text(value):
    """Return empty string for NaN/None, string otherwise."""
    if pd.isna(value):
        return ''
    return str(value).strip()

In [14]:
# ── CELL 4 — Build product context for extraction ────────────────────────────
def build_product_context(row):
    return f"""[PRODUCT_ID]         {safe_text(row['Product ID'])}
[PRODUCT_NAME_CN]    {safe_text(row['product_name_cn'])}
[PRODUCT_NAME_EN]    {safe_text(row['Product Name'])}
[ENGLISH_NAME]       {safe_text(row['English Name'])}
[PRODUCT_TYPE]       {safe_text(row['Product Type'])}
[TARGET_CROPS]       {safe_text(row['Target Crops'])}
[MAIN_INGREDIENTS]   {safe_text(row['Main Ingredients'])}
[DOSAGE]             {safe_text(row['Usage Method and Dosage'])}
[DILUTION_METHOD]    {safe_text(row['Dilution Method'])}
[USAGE_INSTRUCTIONS] {safe_text(row['Usage Instructions'])}"""

In [17]:
# ── CELL 5 — Extraction prompt & function ────────────────────────────────────
SYSTEM_PROMPT = """You are an agricultural product data extraction specialist.

Extract structured entities from the product record below.

FIELD DEFINITIONS:
- target_crops: crops/plants this product is used on
- target_diseases: plant diseases this product treats or prevents
  (fungal, bacterial, viral diseases ONLY — NOT pests, NOT symptoms)
- target_pests: insects, mites, nematodes, molluscs, weeds this product controls
  IMPORTANT: nematodes (root-knot, cyst, stem) are PESTS not diseases
- symptoms: physiological symptoms the product addresses
  (yellowing, dwarfing, mosaic, wilting, curling, tip-drying, cracking, small leaves)
  These are plant stress responses, NOT diseases
- active_ingredients: chemical or biological active ingredients with concentrations
  (e.g. "Bacillus subtilis", "Lambda-cyhalothrin 2.5%", "Thiamethoxam 30%")
- product_types: product category/formulation type
  (e.g. "Pesticide", "Microbial Agent", "Water-Soluble Fertilizer", "Herbicide")
- dosages: application rates and dilution ratios
  (e.g. "1:1000 dilution", "200-300 mL/mu", "800-1000 times dilution")
- application_methods: HOW to apply
  (e.g. "Foliar spray", "Drip irrigation", "Soil drench", "Seed treatment")
- safety_warnings: precautions, restrictions, harvest intervals, toxicity notes

TAGGING RULES:
The [USAGE_INSTRUCTIONS] field contains inline tags from translation:
- [DISEASE] tag → extract that term into target_diseases
- [PEST] tag → extract that term into target_pests
- [SYMPTOM] tag → extract that term into symptoms
Terms without tags: use your judgment based on the field definitions above.

CRITICAL RULES:
- Never place nematodes in target_diseases
- Never place yellowing/dwarfing/mosaic/wilting in target_diseases
- Never invent information not present in the record
- If a field has no relevant information, return []
- Return active_ingredients as a list of individual ingredients, not one long string
- Strip the [DISEASE]/[PEST]/[SYMPTOM] tags from extracted values
- CFU counts, viable bacteria counts, and concentration specs (e.g. "viable bacteria count ≥ 2×10⁸ CFU/ml")
  are NOT separate ingredients — ignore them, the concentration is already implicit in the ingredient name

Return ONLY valid JSON matching this exact schema:
{
  "product_id": "",
  "product_name_cn": "",
  "product_name": "",
  "target_crops": [],
  "target_diseases": [],
  "target_pests": [],
  "symptoms": [],
  "active_ingredients": [],
  "product_types": [],
  "dosages": [],
  "application_methods": [],
  "safety_warnings": []
}"""

def extract_entities(product_context, product_id=''):
    """Extract structured entities from a product context string."""
    try:
        resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': product_context}
        ],
        response_format={'type': 'json_object'},
        temperature=0,
        max_completion_tokens=2048,  # changed from max_tokens
        )
        result = json.loads(resp.choices[0].message.content)

        # Ensure all expected keys exist — fill missing with []
        expected_lists = [
            'target_crops', 'target_diseases', 'target_pests', 'symptoms',
            'active_ingredients', 'product_types', 'dosages',
            'application_methods', 'safety_warnings'
        ]
        for key in expected_lists:
            if key not in result:
                result[key] = []
            elif not isinstance(result[key], list):
                result[key] = [result[key]] if result[key] else []

        return result

    except Exception as e:
        print(f'  ERROR {product_id}: {e}')
        return None


In [18]:
# ── CELL 6 — Single product test before full run ─────────────────────────────
# Always test on 1 product before running all 114
# Test on AF0001 (microbial, has tags), PN0001 (pesticide), AF0035 (blank ingredients)
for test_pid in ['AF0001', 'PN0001', 'AF0035']:
    row = df[df['Product ID'] == test_pid].iloc[0]
    context = build_product_context(row)
    result  = extract_entities(context, product_id=test_pid)
    print(f'\n=== {test_pid} | {row["Product Name"]} ===')
    print(json.dumps(result, indent=2, ensure_ascii=False)[:800])
    print()


=== AF0001 | Citrus Junliqing ===
{
  "product_id": "AF0001",
  "product_name_cn": "柑橘菌立清",
  "product_name": "Citrus Junliqing",
  "target_crops": [
    "chili pepper",
    "citrus",
    "vegetables",
    "fruit trees",
    "melons",
    "field crops",
    "medicinal herbs",
    "flowers",
    "seedlings",
    "tea trees"
  ],
  "target_diseases": [
    "downy mildew",
    "gray mold",
    "purple spot",
    "anthracnose",
    "leaf spot",
    "wilt",
    "root rot",
    "soft rot",
    "ginger wilt",
    "damping-off"
  ],
  "target_pests": [],
  "symptoms": [
    "yellowing",
    "tip-drying",
    "mosaic",
    "dwarfing"
  ],
  "active_ingredients": [
    "Bacillus subtilis",
    "Bacillus licheniformis",
    "Bacillus pumilus",
    "Bacillus gelatinous",
    "Bacillus amyloliquefaciens"
  ],
  "product_types": [
    "


=== PN0001 | Bacillus thuringiensis 4000 IU/μL ===
{
  "product_id": "PN0001",
  "product_name_cn": "苏云金杆菌4000IU/微升",
  "product_name": "Bacillus thuringiensis 40

In [19]:
# ── CELL 7 — Review test output before proceeding ────────────────────────────
# Check the test results above and confirm:
# 1. AF0001: nematodes should be in target_pests not target_diseases
# 2. PN0001: insects should be in target_pests
# 3. AF0035: target_diseases should have diseases (from [DISEASE] tags in instructions)
#            active_ingredients should be [] (blank in source)
# Only proceed to Cell 8 if test results look correct

print('Review test output above.')
print('Confirm categorization is correct before running full extraction.')

Review test output above.
Confirm categorization is correct before running full extraction.


In [20]:
# ── CELL 8 — Full extraction loop ────────────────────────────────────────────
records = []
failed  = []

for idx in tqdm(range(len(df)), desc='Extracting entities'):
    row     = df.iloc[idx]
    pid     = safe_text(row['Product ID'])
    context = build_product_context(row)
    result  = extract_entities(context, product_id=pid)

    if result is None:
        failed.append(pid)
        continue

    # Chinese name from product_name_cn column (recovered in notebook 01 Cell 23)
    result['product_name_cn'] = safe_text(row['product_name_cn'])
    # English name from Product Name (translated in Phase 1)
    result['product_name']    = safe_text(row['Product Name'])

    records.append(result)

print(f'\nExtracted: {len(records)} / {len(df)}')
if failed:
    print(f'Failed:    {failed}')
else:
    print('No failures ✓')

Extracting entities: 100%|██████████| 114/114 [05:00<00:00,  2.63s/it]


Extracted: 114 / 114
No failures ✓


In [21]:
# ── CELL 9 — Build DataFrame and field population report ─────────────────────
entities_df = pd.DataFrame(records)
print(f'Shape: {entities_df.shape}')
print(f'Columns: {entities_df.columns.tolist()}')

print('\n=== FIELD POPULATION REPORT ===')
list_fields = [
    'target_crops', 'target_diseases', 'target_pests', 'symptoms',
    'active_ingredients', 'product_types', 'dosages',
    'application_methods', 'safety_warnings'
]
for field in list_fields:
    populated   = entities_df[field].apply(lambda x: len(x) > 0).sum()
    empty       = len(entities_df) - populated
    pct         = populated / len(entities_df) * 100
    print(f'  {field:<25} populated={populated:>3}  empty={empty:>3}  ({pct:.1f}%)')


Shape: (114, 12)
Columns: ['product_id', 'product_name_cn', 'product_name', 'target_crops', 'target_diseases', 'target_pests', 'symptoms', 'active_ingredients', 'product_types', 'dosages', 'application_methods', 'safety_warnings']

=== FIELD POPULATION REPORT ===
  target_crops              populated=113  empty=  1  (99.1%)
  target_diseases           populated= 34  empty= 80  (29.8%)
  target_pests              populated= 62  empty= 52  (54.4%)
  symptoms                  populated= 46  empty= 68  (40.4%)
  active_ingredients        populated=100  empty= 14  (87.7%)
  product_types             populated=114  empty=  0  (100.0%)
  dosages                   populated=112  empty=  2  (98.2%)
  application_methods       populated=101  empty= 13  (88.6%)
  safety_warnings           populated=100  empty= 14  (87.7%)


In [22]:
# ── CELL 10 — Categorization audit ───────────────────────────────────────────
print('=== NEMATODES IN target_diseases (should be 0) ===')
nematode_kw = ['nematode', 'nematodes']
problems = []
for _, row in entities_df.iterrows():
    for d in row['target_diseases']:
        if any(k in d.lower() for k in nematode_kw):
            problems.append(f"  {row['product_id']}: '{d}'")
if problems:
    for p in problems: print(p)
else:
    print('  None found ✓')

print('\n=== SYMPTOM-LIKE TERMS IN target_diseases (should be 0) ===')
symptom_kw = ['yellowing', 'dwarfing', 'mosaic', 'wilting', 'curling',
              'tip-drying', 'cracking', 'small leaves', 'chlorosis']
problems = []
for _, row in entities_df.iterrows():
    for d in row['target_diseases']:
        if any(k in d.lower() for k in symptom_kw):
            problems.append(f"  {row['product_id']}: '{d}'")
if problems:
    for p in problems: print(p)
else:
    print('  None found ✓')

print('\n=== VAGUE GENERIC TERMS (should be 0) ===')
vague = ['bacteria', 'fungi', 'viruses', 'various viruses', 'pathogenic',
         'physiological', 'soil-borne', 'fungal diseases', 'bacterial diseases']
problems = []
for _, row in entities_df.iterrows():
    for d in row['target_diseases']:
        if any(k in d.lower() for k in vague):
            problems.append(f"  {row['product_id']}: '{d}'")
if problems:
    for p in problems: print(p)
else:
    print('  None found ✓')

print('\n=== AF0012 NEMATODE PRODUCT CHECK ===')
af12 = entities_df[entities_df['product_id'] == 'AF0012'].iloc[0]
print(f'  target_diseases: {af12["target_diseases"]}')
print(f'  target_pests:    {af12["target_pests"]}')

=== NEMATODES IN target_diseases (should be 0) ===
  None found ✓

=== SYMPTOM-LIKE TERMS IN target_diseases (should be 0) ===
  AF0024: 'fruit cracking'
  AF0026: 'cracking'

=== VAGUE GENERIC TERMS (should be 0) ===
  AF0003: 'Bacterial Wilt'
  AF0006: 'Bacterial Wilt'
  AF0007: 'Bacterial Wilt'
  AF0009: 'Bacterial Wilt'
  AF0010: 'Fungal diseases'
  AF0010: 'Bacterial diseases'
  AF0011: 'Bacterial Wilt'
  AF0033: 'soil-borne diseases'
  AF0035: 'Bacterial Gummosis'
  AF0036: 'physiological diseases'
  AF0036: 'soil-borne diseases'
  PN0029: 'Cucumber Bacterial Angular Leaf Spot'
  PN0029: 'Peach Bacterial Spot'
  PN0029: 'Rice Bacterial Stripe Disease'
  AF0042: 'fungal diseases'
  AF0042: 'bacterial diseases'
  AF0042: 'soil-borne diseases'
  AF0052: 'fungal diseases'
  AF0052: 'bacterial diseases'
  AF0056: 'viruses'
  AF0060: 'Fungal diseases'
  AF0060: 'Bacterial diseases'

=== AF0012 NEMATODE PRODUCT CHECK ===
  target_diseases: ['root rot', 'stem rot', 'ginger wilt', 'dampin

In [23]:
# Direct fix for cracking misclassification
for pid, term in [('AF0024', 'fruit cracking'), ('AF0026', 'cracking')]:
    idx = entities_df[entities_df['product_id'] == pid].index[0]
    # Remove from target_diseases
    entities_df.at[idx, 'target_diseases'] = [
        d for d in entities_df.at[idx, 'target_diseases'] if d != term
    ]
    # Add to symptoms if not already there
    if term not in entities_df.at[idx, 'symptoms']:
        entities_df.at[idx, 'symptoms'] = entities_df.at[idx, 'symptoms'] + [term]
    print(f'Fixed {pid}: moved "{term}" → symptoms')

Fixed AF0024: moved "fruit cracking" → symptoms
Fixed AF0026: moved "cracking" → symptoms


In [24]:
REMOVE_VAGUE = {
    'fungal diseases', 'bacterial diseases', 'soil-borne diseases',
    'physiological diseases', 'fungal diseases', 'various viruses',
    'viruses'
}
count = 0
for idx in entities_df.index:
    original = entities_df.at[idx, 'target_diseases']
    cleaned  = [d for d in original if d.lower() not in REMOVE_VAGUE]
    if len(cleaned) != len(original):
        removed = set(original) - set(cleaned)
        print(f"  {entities_df.at[idx, 'product_id']}: removed {removed}")
        entities_df.at[idx, 'target_diseases'] = cleaned
        count += 1
print(f'\nCleaned {count} products')

  AF0010: removed {'Bacterial diseases', 'Fungal diseases'}
  AF0033: removed {'soil-borne diseases'}
  AF0036: removed {'soil-borne diseases', 'physiological diseases'}
  AF0042: removed {'soil-borne diseases', 'fungal diseases', 'bacterial diseases'}
  AF0052: removed {'fungal diseases', 'bacterial diseases'}
  AF0056: removed {'viruses'}
  AF0060: removed {'Bacterial diseases', 'Fungal diseases'}

Cleaned 7 products


In [25]:
# Verify fixes
print('AF0024 symptoms:', entities_df[entities_df['product_id']=='AF0024'].iloc[0]['symptoms'])
print('AF0024 diseases:', entities_df[entities_df['product_id']=='AF0024'].iloc[0]['target_diseases'])
print('AF0026 symptoms:', entities_df[entities_df['product_id']=='AF0026'].iloc[0]['symptoms'])
print('AF0010 diseases:', entities_df[entities_df['product_id']=='AF0010'].iloc[0]['target_diseases'])
print('AF0042 diseases:', entities_df[entities_df['product_id']=='AF0042'].iloc[0]['target_diseases'])

AF0024 symptoms: ['leaf chlorosis', 'fruit cracking']
AF0024 diseases: ['blossom-end rot']
AF0026 symptoms: ['stunted fruit', 'green tip fruit', 'leaf curling', 'chlorosis', 'growth stagnation', 'cracking']
AF0010 diseases: ['Ulcer disease', 'Gummosis', 'Black spot', 'Rot', 'Moss', 'Viral diseases']
AF0042 diseases: ['root rot']


In [26]:
# Remove 'Viral diseases' from AF0010 — too vague, source says fungal/bacterial
idx = entities_df[entities_df['product_id'] == 'AF0010'].index[0]
entities_df.at[idx, 'target_diseases'] = [
    d for d in entities_df.at[idx, 'target_diseases']
    if d.lower() != 'viral diseases'
]
print('AF0010 diseases after fix:', entities_df[entities_df['product_id']=='AF0010'].iloc[0]['target_diseases'])

# Also check AF0012 viral diseases — same issue
idx12 = entities_df[entities_df['product_id'] == 'AF0012'].index[0]
print('AF0012 diseases:', entities_df[entities_df['product_id']=='AF0012'].iloc[0]['target_diseases'])

AF0010 diseases after fix: ['Ulcer disease', 'Gummosis', 'Black spot', 'Rot', 'Moss']
AF0012 diseases: ['root rot', 'stem rot', 'ginger wilt', 'damping-off', 'viral diseases']


In [27]:
# ── CELL 11 — Save product_entities.json ─────────────────────────────────────
records_out = entities_df.to_dict(orient='records')
with open('product_entities.json', 'w', encoding='utf-8') as f:
    json.dump(records_out, f, ensure_ascii=False, indent=2)
print(f'Saved → product_entities.json  ({len(records_out)} products)')

# Verify save
with open('product_entities.json', encoding='utf-8') as f:
    verify = json.load(f)
print(f'Verified: {len(verify)} records loaded back ✓')
print(f'Keys: {list(verify[0].keys())}')
print(f'\nSpot check CN names:')
for pid in ['AF0001', 'PN0001', 'AF0035']:
    rec = next(r for r in verify if r['product_id'] == pid)
    print(f"  {pid}: CN='{rec['product_name_cn']}' | EN='{rec['product_name']}'")

Saved → product_entities.json  (114 products)
Verified: 114 records loaded back ✓
Keys: ['product_id', 'product_name_cn', 'product_name', 'target_crops', 'target_diseases', 'target_pests', 'symptoms', 'active_ingredients', 'product_types', 'dosages', 'application_methods', 'safety_warnings']

Spot check CN names:
  AF0001: CN='柑橘菌立清' | EN='Citrus Junliqing'
  PN0001: CN='苏云金杆菌4000IU/微升' | EN='Bacillus thuringiensis 4000 IU/μL'
  AF0035: CN='百菌灵' | EN='Carbendazim'


In [31]:
# ── CELL 12 — Save, download, done ───────────────────────────────────────────
from google.colab import files

# Save JSON — primary format for downstream notebooks
with open('product_entities_v2.json', 'w', encoding='utf-8') as f:
    json.dump(records_out, f, ensure_ascii=False, indent=2)
print('Saved → product_entities_v2.json')

# Save CSV — for human review in Excel
entities_df.to_csv('product_entities_v2.csv', index=False, encoding='utf-8-sig')
print('Saved → product_entities_v2.csv')

# Download both
files.download('product_entities_v2.json')
files.download('product_entities_v2.csv')

print("""
=== PHASE 2 COMPLETE ✓ ===
Input:  ProductCatalog_TRANSLATED_v2.xlsx
Output: product_entities_v2.json
        product_entities_v2.csv
114 products | 12 fields | CN + EN names
Next: notebook_03_normalization.ipynb
""")

Saved → product_entities_v2.json  (114 products)
Saved → product_entities_v2.csv

AF0012 pests: ['root-knot nematodes', 'cyst nematodes', 'stem nematodes', 'pest eggs', 'overwintering pests']
AF0042 ingredients: ['Microbial agent', 'Effective viable bacteria count ≥3×10⁹/ml']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


=== PHASE 2 COMPLETE ✓ ===
Input:  ProductCatalog_TRANSLATED_v2.xlsx
Output: product_entities_v2.json
        product_entities_v2.csv
114 products | 12 fields | CN + EN names
Next: notebook_03_normalization.ipynb



In [32]:
# ── Apply all fixes + save in one cell ───────────────────────────────────────

# Fix 1: Remove CFU spec from AF0042
idx = entities_df[entities_df['product_id'] == 'AF0042'].index[0]
entities_df.at[idx, 'active_ingredients'] = [
    i for i in entities_df.at[idx, 'active_ingredients']
    if 'viable' not in i.lower() and 'count' not in i.lower()
]

# Fix 2: Remove vague pest terms from ALL products
VAGUE_PESTS = {'pest eggs', 'overwintering pests'}
for idx in entities_df.index:
    original = entities_df.at[idx, 'target_pests']
    cleaned  = [p for p in original if p.lower() not in VAGUE_PESTS]
    if len(cleaned) != len(original):
        entities_df.at[idx, 'target_pests'] = cleaned

# Verify in memory before saving
af12 = entities_df[entities_df['product_id'] == 'AF0012'].iloc[0]
af42 = entities_df[entities_df['product_id'] == 'AF0042'].iloc[0]
print('AF0012 pests:', af12['target_pests'])
print('AF0042 ingredients:', af42['active_ingredients'])

# Save
records_out = entities_df.to_dict(orient='records')
with open('product_entities_v2.json', 'w', encoding='utf-8') as f:
    json.dump(records_out, f, ensure_ascii=False, indent=2)
entities_df.to_csv('product_entities_v2.csv', index=False, encoding='utf-8-sig')
print(f'\nSaved {len(records_out)} products')

from google.colab import files
files.download('product_entities_v2.json')
files.download('product_entities_v2.csv')

AF0012 pests: ['root-knot nematodes', 'cyst nematodes', 'stem nematodes']
AF0042 ingredients: ['Microbial agent']

Saved 114 products


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>